# Fase 2 · Entrenamos un clasificador de opiniones

Continuamos la fase 1 con un conjunto mayor: **60 opiniones del archivo `dataset_opiniones_60.csv`**. Vamos a entrenar un modelo para predecir si un comentario es positivo o negativo.

Utilizaremos Pandas para trabajar con la tabla y Scikit-learn para separar los datos, convertir los textos con TF-IDF, entrenar una regresión logística y evaluar sus predicciones.

Este notebook es independiente: no necesitas ejecutar antes la fase 1. Ejecuta las celdas de arriba abajo.

## Antes de empezar

Guarda este notebook y el CSV en la misma carpeta. En VS Code, selecciona un kernel de Python. En Colab, sube el CSV al panel Archivos antes de ejecutar.

Si necesitas instalar las dependencias, descomenta y ejecuta la siguiente celda. Después, reinicia el kernel si te lo pide.

In [ ]:
# Ejecuta esta línea solo si faltan las librerías.
%pip install pandas scikit-learn

## 1. Cargamos las opiniones

`pd.read_csv()` lee el CSV y crea un DataFrame. La columna `comentario` contiene el texto y `sentimiento` contiene la respuesta conocida. La columna `id` identifica cada opinión: no la usaremos para entrenar.

In [1]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier

df = pd.read_csv("dataset_opiniones_60.csv", encoding="utf-8-sig")

# Mostramos cinco opiniones para comprobar que se han cargado.
display(df.head())
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

,id,comentario,sentimiento
0,1,La experiencia ha sido muy positiva.,positivo
1,2,Me atendieron con poca paciencia y sin amabili...,negativo
2,3,Recibí exactamente lo que había pedido.,positivo
3,4,Funciona de maravilla y cumple lo que promete.,positivo
4,5,El pedido llegó antes de lo esperado.,positivo


Filas: 60 | Columnas: 3


## 2. Revisamos los datos antes de entrenar

Comprobamos los valores vacíos, las etiquetas y los comentarios repetidos. Un comentario duplicado podría aparecer tanto en entrenamiento como en prueba y dar una evaluación demasiado optimista.

No quitamos palabras como **«no»**, porque una negación puede cambiar el sentimiento. TF-IDF convierte a minúsculas de forma predeterminada; no necesitamos hacerlo aparte.

In [2]:
# Revisamos vacíos por columna.
display(df.isna().sum().to_frame("valores_vacios"))

# Contamos los comentarios de cada sentimiento.
display(df["sentimiento"].value_counts().to_frame("opiniones"))
print("Comentarios repetidos:", df["comentario"].duplicated().sum())

# Estos controles detienen la ejecución si los datos no son adecuados.
assert not df[["comentario", "sentimiento"]].isna().any().any(), "Revisa los valores vacíos."
assert df["comentario"].str.strip().ne("").all(), "Hay comentarios sin texto."
assert set(df["sentimiento"]) == {"positivo", "negativo"}, "Revisa las etiquetas."
assert not df["comentario"].duplicated().any(), "Revisa los comentarios repetidos."
print("Datos preparados para continuar.")

,valores_vacios
id,0
comentario,0
sentimiento,0


,opiniones
sentimiento,
positivo,30
negativo,30


Comentarios repetidos: 0
Datos preparados para continuar.


## 3. Separamos entrenamiento y prueba

- `X`: comentarios originales, todavía como texto.
- `y`: sentimientos conocidos, que el modelo debe aprender a predecir.
- `test_size=0.2`: reservamos el 20 % para prueba; el resto sirve para entrenar.
- `random_state=42`: fijamos el azar para repetir la misma separación con los mismos datos.
- `stratify=y`: mantenemos la proporción de positivos y negativos en ambos grupos.

**Primero separamos y después calculamos TF-IDF.** Así los textos de prueba no intervienen al aprender el vocabulario ni sus pesos. No reutilizamos la matriz TF-IDF calculada con todos los datos en la fase 1.

In [3]:
X = df["comentario"]
y = df["sentimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Opiniones para entrenar: {len(X_train)}")
print(f"Opiniones para probar: {len(X_test)}")
display(pd.DataFrame({
    "entrenamiento": y_train.value_counts(),
    "prueba": y_test.value_counts()
}))

,entrenamiento,prueba
sentimiento,,
negativo,24,6
positivo,24,6


Opiniones para entrenar: 48
Opiniones para probar: 12


## 4. Creamos el proceso y entrenamos el modelo

Un `Pipeline` une dos pasos:

1. `TfidfVectorizer()` aprende el vocabulario del entrenamiento y convierte los comentarios en números.
2. `LogisticRegression()` aprende relaciones entre esos números y las etiquetas positivas o negativas.

Los nombres `"tfidf"` y `"clasificador"` nos permiten acceder después a cada paso. `fit(X_train, y_train)` entrena el proceso completo. Aunque se llama regresión logística, aquí realiza una tarea de **clasificación**.

`max_iter=1000` establece un máximo de iteraciones para que el algoritmo encuentre una solución; no significa que recorra exactamente mil veces todos los comentarios.

In [4]:
modelo = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clasificador", LogisticRegression(max_iter=1000, random_state=42))
])

# Solo usamos los datos de entrenamiento para aprender.
modelo.fit(X_train, y_train)
print("Modelo entrenado.")

Modelo entrenado.


### Vemos una muestra de los números utilizados

Accedemos al vectorizador ya entrenado. `transform()` utiliza el vocabulario aprendido, sin volver a entrenarlo. Cada fila representa un comentario y cada columna una palabra; cero significa que esa palabra no aparece.

Mostramos solo tres comentarios y doce palabras para que la tabla sea legible. En conjuntos grandes conviene mantener la matriz dispersa y evitar convertirla entera con `toarray()`.

In [5]:
vectorizador = modelo.named_steps["tfidf"]
matriz_muestra = vectorizador.transform(X_train.iloc[:3])
palabras = vectorizador.get_feature_names_out()

df_tfidf_muestra = pd.DataFrame(
    matriz_muestra[:, :12].toarray(),
    columns=palabras[:12],
    index=X_train.iloc[:3].index
)
print(f"Palabras del vocabulario de entrenamiento: {len(palabras)}")
display(df_tfidf_muestra.round(3))

,abierto,acertada,al,amabilidad,amable,antes,arrepiento,artículo,atención,atendieron,avisó,bien
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Palabras del vocabulario de entrenamiento: 138


## 5. Evaluamos con opiniones que no ha visto

`predict(X_test)` convierte los textos de prueba con el TF-IDF aprendido y predice sus etiquetas. Después las comparamos con las respuestas reales.

`accuracy_score()` calcula la **exactitud**: la proporción total de aciertos. No es lo mismo que la métrica `precision` del informe.

In [6]:
predicciones = modelo.predict(X_test)
exactitud = accuracy_score(y_test, predicciones)

resultados = pd.DataFrame({
    "comentario": X_test,
    "sentimiento_real": y_test,
    "prediccion": predicciones
})
resultados["acierto"] = resultados["sentimiento_real"] == resultados["prediccion"]

display(resultados)
print(f"Aciertos: {resultados['acierto'].sum()} de {len(resultados)}")
print(f"Exactitud: {exactitud:.2%}")

,comentario,sentimiento_real,prediccion,acierto
27,No recomiendo esta tienda a nadie.,negativo,negativo,True
58,"El producto es cómodo, práctico y fácil de uti...",positivo,negativo,False
21,"Me encanta el producto, funciona perfectamente.",positivo,negativo,False
20,La experiencia ha sido muy negativa.,negativo,positivo,False
22,El equipo resolvió mi problema enseguida.,positivo,negativo,False
34,No volvería a comprar en esta tienda.,negativo,negativo,True
12,Todo el proceso fue sencillo y agradable.,positivo,negativo,False
24,La solución que me ofrecieron no sirvió de nada.,negativo,negativo,True
49,El servicio fue rápido y eficiente.,positivo,positivo,True
44,El servicio fue lento y poco eficiente.,negativo,positivo,False


Aciertos: 4 de 12
Exactitud: 33.33%


### Comparamos con una referencia sencilla

Un modelo que siempre responde la clase más frecuente puede acertar sin aprender nada sobre el texto. Usamos `DummyClassifier` para comprobar si nuestro modelo mejora esa referencia.

In [7]:
referencia = DummyClassifier(strategy="most_frequent")
referencia.fit(X_train.to_frame(), y_train)
predicciones_referencia = referencia.predict(X_test.to_frame())
exactitud_referencia = accuracy_score(y_test, predicciones_referencia)

print(f"Exactitud de referencia: {exactitud_referencia:.2%}")
print(f"Exactitud de regresión logística: {exactitud:.2%}")

Exactitud de referencia: 50.00%
Exactitud de regresión logística: 33.33%


### Informe y matriz de confusión

- **Precision:** de las predicciones de una clase, qué proporción fue correcta.
- **Recall:** de los casos reales de esa clase, qué proporción detectamos.
- **F1-score:** combina precision y recall.
- **Support:** número de opiniones reales de cada clase en la prueba.

En la matriz de confusión, las **filas son las etiquetas reales** y las **columnas son las predichas**. La diagonal contiene los aciertos; fuera de ella están los errores. `zero_division=0` muestra cero cuando una métrica no se puede calcular por falta de predicciones de una clase.

In [8]:
etiquetas = ["negativo", "positivo"]
print(classification_report(
    y_test, predicciones, labels=etiquetas, zero_division=0
))

matriz = confusion_matrix(y_test, predicciones, labels=etiquetas)
display(pd.DataFrame(
    matriz,
    index=["Real: negativo", "Real: positivo"],
    columns=["Predicho: negativo", "Predicho: positivo"]
))

,Predicho: negativo,Predicho: positivo
Real: negativo,3,3
Real: positivo,5,1


              precision    recall  f1-score   support

    negativo       0.38      0.50      0.43         6
    positivo       0.25      0.17      0.20         6

    accuracy                           0.33        12
   macro avg       0.31      0.33      0.31        12
weighted avg       0.31      0.33      0.31        12



## 6. Revisamos los errores

Mostramos las opiniones cuya predicción no coincide con la etiqueta real. Busca negaciones, palabras desconocidas o expresiones parecidas con sentimientos opuestos.

TF-IDF con sus opciones predeterminadas trabaja con palabras individuales: no representa directamente el orden de una frase como «no funciona». Esto puede dificultar la clasificación de algunas opiniones.

In [9]:
errores = resultados.loc[~resultados["acierto"]]
print(f"Errores encontrados: {len(errores)}")
display(errores)

,comentario,sentimiento_real,prediccion,acierto
58,"El producto es cómodo, práctico y fácil de uti...",positivo,negativo,False
21,"Me encanta el producto, funciona perfectamente.",positivo,negativo,False
20,La experiencia ha sido muy negativa.,negativo,positivo,False
22,El equipo resolvió mi problema enseguida.,positivo,negativo,False
12,Todo el proceso fue sencillo y agradable.,positivo,negativo,False
44,El servicio fue lento y poco eficiente.,negativo,positivo,False
56,Estoy muy contenta con mi compra.,positivo,negativo,False
45,La experiencia de compra ha sido horrible.,negativo,positivo,False


Errores encontrados: 8


## 7. Probamos con nuevas opiniones

Ahora introducimos textos propios. Pasamos texto al Pipeline: no hace falta convertirlo manualmente a números.

Estas predicciones sirven para explorar el comportamiento del modelo. Sin etiquetas reales verificadas, no son una nueva medida de exactitud.

In [10]:
nuevas_opiniones = [
    "Estoy feliz con la compra, todo funciona de maravilla",
    "El paquete llegó destrozado y nadie me ayudó",
    "No estoy contenta con el producto",
    "Me encanta, el producto es maravilloso",
    "El pedido llegó tarde y estoy muy enfadada",
    "La calidad es estupenda y lo recomiendo",

]

predicciones_nuevas = modelo.predict(nuevas_opiniones)
display(pd.DataFrame({
    "comentario": nuevas_opiniones,
    "sentimiento_predicho": predicciones_nuevas
}))

,comentario,sentimiento_predicho
0,"Estoy feliz con la compra, todo funciona de ma...",positivo
1,El paquete llegó destrozado y nadie me ayudó,negativo
2,No estoy contenta con el producto,negativo
3,"Me encanta, el producto es maravilloso",negativo
4,El pedido llegó tarde y estoy muy enfadada,negativo
5,La calidad es estupenda y lo recomiendo,positivo


## 8. Conclusiones de nuestra ejecución

La siguiente celda resume los resultados realmente obtenidos. No debemos asumir que añadir más opiniones garantiza automáticamente mejores predicciones: también importan su variedad, su calidad y cómo representan los casos futuros.

In [11]:
print(f"Hemos utilizado {len(df)} opiniones etiquetadas.")
print(f"Entrenamiento: {len(X_train)} opiniones. Prueba: {len(X_test)} opiniones.")
print(f"El modelo acertó {resultados['acierto'].sum()} de {len(X_test)} opiniones de prueba.")
print(f"Exactitud del modelo: {exactitud:.2%}; referencia: {exactitud_referencia:.2%}.")
if exactitud > exactitud_referencia:
    print("En esta separación, el modelo mejora la referencia de la clase más frecuente.")
else:
    print("En esta separación, el modelo no mejora la referencia: debemos revisar sus errores.")
print("Con solo 12 opiniones de prueba, cada acierto cambia el resultado unos 8,33 puntos porcentuales.")

Hemos utilizado 60 opiniones etiquetadas.
Entrenamiento: 48 opiniones. Prueba: 12 opiniones.
El modelo acertó 4 de 12 opiniones de prueba.
Exactitud del modelo: 33.33%; referencia: 50.00%.
En esta separación, el modelo no mejora la referencia: debemos revisar sus errores.
Con solo 12 opiniones de prueba, cada acierto cambia el resultado unos 8,33 puntos porcentuales.


### Qué hemos aprendido y qué podemos mejorar

Hemos cargado un CSV, revisado sus datos, separado entrenamiento y prueba, entrenado un Pipeline y evaluado sus predicciones.

El conjunto tiene solo 60 opiniones y varias frases con estructuras parecidas. El resultado de esta pequeña prueba no garantiza el mismo rendimiento con clientes reales. El modelo encuentra patrones estadísticos; no comprende el texto como una persona.

Como siguiente experimento podemos recoger más opiniones variadas y probar `ngram_range=(1, 2)` para incluir parejas de palabras, como «no funciona». Si usamos esta prueba repetidamente para elegir cambios, deja de ser una evaluación independiente: para comparar versiones conviene utilizar validación cruzada sobre el entrenamiento y reservar una prueba final aparte.

**Para practicar:** explica un error del modelo, añade tres comentarios propios y razona qué etiqueta esperarías antes de ejecutar la predicción.